# PhishCatcher Training Pipeline

This notebook provides an interactive exploration and training workflow for the phishing detection system.

## Modules
- `ml.preprocessing` - Text cleaning
- `ml.feature_engineering` - URL, text, structural features
- `ml.models.classical` - TF-IDF + LR/SVM/XGBoost
- `ml.models.bert_model` - DistilBERT embeddings (future feature)
- `ml.evaluation` - Metrics & SHAP/LIME

In [ ]:
import sys
sys.path.insert(0, '../app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json
warnings.filterwarnings('ignore')

## 1. Load Data & EDA

In [ ]:
# Load archive dataset
archive_path = Path('../archive/phishing_email.csv')
df = pd.read_csv(archive_path)
print(f"Total emails: {len(df)}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nPhishing: {df['label'].sum()}, Legitimate: {len(df) - df['label'].sum()}")

In [ ]:
# EDA: Email length distribution
df['text_length'] = df['text_combined'].astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(df['text_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Email Length Distribution')
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f"Mean: {df['text_length'].mean():.0f}")
axes[0].legend()

# By class
for label, color, name in [(0, 'green', 'Legitimate'), (1, 'red', 'Phishing')]:
    subset = df[df['label'] == label]
    axes[1].hist(subset['text_length'], bins=30, alpha=0.5, label=f"{name} (n={len(subset)})", color=color)
axes[1].set_title('Length Distribution by Class')
axes[1].legend()
axes[1].set_xlabel('Text Length (characters)')

plt.tight_layout()
plt.show()

# Statistics by class
print("\n--- Text Length Statistics ---")
print(df.groupby('label')['text_length'].describe())

## 2. Preprocessing

In [ ]:
from ml.preprocessing import EmailPreprocessor

processor = EmailPreprocessor()

# Sample for quick testing
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42)

print("Preprocessing emails...")
df_sample['processed_text'] = df_sample['text_combined'].astype(str).apply(processor.preprocess_text)
print(f"Done! Preprocessed {len(df_sample)} emails")

# Show example
print("\n--- Example Preprocessing ---")
print(f"Original: {df_sample['text_combined'].iloc[0][:200]}...")
print(f"\nProcessed: {df_sample['processed_text'].iloc[0][:200]}...")

## 3. Feature Engineering

In [ ]:
from ml.feature_engineering import FeatureEngineer

engineer = FeatureEngineer()

print("Extracting features from sample...")
features_list = []

for idx, row in df_sample.head(100).iterrows():
    text = str(row['text_combined'])
    lines = text.split('\n')[:15]
    subject = ' '.join(lines[:2])[:200]
    body = ' '.join(lines[2:])[:3000]
    
    features = engineer.extract_all_features(subject, body)
    features_list.append(features)

features_df = pd.DataFrame(features_list)
print(f"Extracted {len(features_list)} samples with {len(features_df.columns)} features")
print(f"\nFeature names: {features_df.columns.tolist()[:10]}...")

## 4. Model Comparison

In [ ]:
# Load saved model results
model_dir = Path('../models')

# Classical ML results
classical_path = model_dir / 'classical_ml_results.json'
if classical_path.exists():
    with open(classical_path, 'r') as f:
        classical_results = json.load(f)
    print("=== CLASSICAL ML MODELS ===")
    for name, metrics in classical_results.items():
        print(f"{name}: Accuracy={metrics['accuracy']:.4f}, F1={metrics['f1_score']:.4f}")

# Training metrics
training_path = model_dir / 'training_metrics.json'
if training_path.exists():
    with open(training_path, 'r') as f:
        training_results = json.load(f)
    print("\n=== TF-IDF BASED MODELS ===")
    if 'text_classifier' in training_results:
        tc = training_results['text_classifier']
        print(f"Text Classifier: Accuracy={tc['accuracy']:.4f}, F1={tc['f1_score']:.4f}")
    if 'feature_detector' in training_results:
        fd = training_results['feature_detector']
        print(f"Feature Detector: Accuracy={fd['accuracy']:.4f}, F1={fd['f1_score']:.4f}")

In [ ]:
# Visualize model comparison
import matplotlib.pyplot as plt
import numpy as np

models = ['Logistic\nRegression', 'SVM', 'XGBoost', 'Text\nClassifier', 'Feature\nDetector']
accuracy = [0.964, 0.968, 0.958, 0.902, 0.682]
f1_scores = [0.966, 0.970, 0.961, 0.906, 0.633]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, accuracy, width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1 Score', color='coral')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.5, 1.0)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("\nBest Model: SVM with 96.8% accuracy and 0.97 F1 score")

## 5. Quick Inference Test

In [ ]:
from ml.text_classifier import get_text_classifier

# Load model
classifier = get_text_classifier()

# Test phishing email
phishing_subject = 'URGENT: Your Account Will Be Suspended'
phishing_body = 'Click here immediately to verify your account. Failure to verify will result in permanent account closure.'

phishing_result = classifier.predict_proba(phishing_subject, phishing_body)

# Test legitimate email
legit_subject = 'Team Meeting Tomorrow'
legit_body = 'Hi team, Just a reminder about our meeting tomorrow at 2 PM. Please come prepared with your updates.'

legit_result = classifier.predict_proba(legit_subject, legit_body)

print("=== PREDICTION TESTS ===")
print(f"\nPhishing Email:")
print(f"  Subject: {phishing_subject}")
print(f"  Probability: {phishing_result['phishing_probability']:.4f}")
print(f"  Category: {phishing_result['category']}")
print(f"  Confidence: {phishing_result['confidence']:.4f}")

print(f"\nLegitimate Email:")
print(f"  Subject: {legit_subject}")
print(f"  Probability: {legit_result['phishing_probability']:.4f}")
print(f"  Category: {legit_result['category']}")
print(f"  Confidence: {legit_result['confidence']:.4f}")

## 6. API Usage (for production)

In [ ]:
# To use the FastAPI endpoint, start the server:
# cd ../app
# uvicorn main:app --reload
# 
# Then send POST request to /ml/predict:
# 
# import requests
# response = requests.post(
#     "http://localhost:8000/ml/predict",
#     json={
#         "subject": "Test Subject",
#         "body": "Test body content"
#     }
# )
# print(response.json())